In [1]:
import cv2
from ultralytics import YOLO
from collections import defaultdict
#load the yolo model
model=YOLO("yolo11n.pt") 

In [2]:
class_list=model.names
class_list

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [ ]:
cap  

In [ ]:
import cv2
from ultralytics import YOLO

# ==========================================
# LOAD YOLO MODEL
# ==========================================

model = YOLO("yolo11n.pt")


# ==========================================
# VIDEO PATH
# ==========================================

video_path = r"D:\vehicletraking\131232-749706873.mp4"

# Example Windows path:
# video_path = r"D:\vehicle_detection\traffic.mp4"


# ==========================================
# OPEN VIDEO
# ==========================================

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Video could not be opened")
    exit()


# ==========================================
# VEHICLE CLASSES
# COCO DATASET
# ==========================================

vehicle_classes = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}


# ==========================================
# COUNTING LINES
# ==========================================

UP_LINE = 250
DOWN_LINE = 400


# ==========================================
# TRACKING DATA
# ==========================================

previous_positions = {}

crossed_down_line = set()
crossed_up_line = set()

counted_up = set()
counted_down = set()


# ==========================================
# TOTAL COUNT
# ==========================================

up_count = 0
down_count = 0


# ==========================================
# VEHICLE-WISE COUNT
# ==========================================

up_vehicle = {
    "Car": 0,
    "Motorcycle": 0,
    "Bus": 0,
    "Truck": 0
}

down_vehicle = {
    "Car": 0,
    "Motorcycle": 0,
    "Bus": 0,
    "Truck": 0
}


# ==========================================
# VIDEO LOOP
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        break


    # ======================================
    # YOLO TRACKING
    # ======================================

    results = model.track(
        frame,
        persist=True,
        classes=[2, 3, 5, 7],
        conf=0.4,
        verbose=False
    )


    # ======================================
    # DRAW TWO LINES
    # ======================================

    # RED = UP LINE
    cv2.line(
        frame,
        (0, UP_LINE),
        (frame.shape[1], UP_LINE),
        (0, 0, 255),
        3
    )


    # BLUE = DOWN LINE
    cv2.line(
        frame,
        (0, DOWN_LINE),
        (frame.shape[1], DOWN_LINE),
        (255, 0, 0),
        3
    )


    cv2.putText(
        frame,
        "UP LINE",
        (20, UP_LINE - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )


    cv2.putText(
        frame,
        "DOWN LINE",
        (20, DOWN_LINE - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )


    # ======================================
    # CHECK TRACKING IDS
    # ======================================

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        class_ids = results[0].boxes.cls.cpu().numpy()

        track_ids = results[0].boxes.id.cpu().numpy()


        # ==================================
        # PROCESS EACH VEHICLE
        # ==================================

        for box, class_id, track_id in zip(
            boxes,
            class_ids,
            track_ids
        ):

            class_id = int(class_id)

            track_id = int(track_id)


            # Ignore non-vehicle objects

            if class_id not in vehicle_classes:
                continue


            vehicle_name = vehicle_classes[class_id]


            # ==================================
            # BOUNDING BOX
            # ==================================

            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ==================================
            # CENTER POINT
            # ==================================

            center_x = int(
                (x1 + x2) / 2
            )

            center_y = int(
                (y1 + y2) / 2
            )


            # Draw center point

            cv2.circle(
                frame,
                (center_x, center_y),
                5,
                (0, 255, 255),
                -1
            )


            # ==================================
            # PREVIOUS Y POSITION
            # ==================================

            previous_y = previous_positions.get(
                track_id
            )


            # ==================================
            # MOVEMENT LOGIC
            # ==================================

            if previous_y is not None:

                # --------------------------------
                # Vehicle moving DOWN → UP
                # --------------------------------

                if (
                    previous_y > DOWN_LINE
                    and center_y <= DOWN_LINE
                ):

                    crossed_down_line.add(
                        track_id
                    )


                # --------------------------------
                # Vehicle moving UP → DOWN
                # --------------------------------

                if (
                    previous_y < UP_LINE
                    and center_y >= UP_LINE
                ):

                    crossed_up_line.add(
                        track_id
                    )


                # =================================
                # UPWARD VEHICLE
                # =================================

                if (
                    track_id in crossed_down_line
                    and center_y < UP_LINE
                    and track_id not in counted_up
                ):

                    up_count += 1

                    up_vehicle[
                        vehicle_name
                    ] += 1

                    counted_up.add(
                        track_id
                    )

                    print(
                        f"UPWARD -> "
                        f"{vehicle_name} "
                        f"ID: {track_id}"
                    )


                # =================================
                # DOWNWARD VEHICLE
                # =================================

                if (
                    track_id in crossed_up_line
                    and center_y > DOWN_LINE
                    and track_id not in counted_down
                ):

                    down_count += 1

                    down_vehicle[
                        vehicle_name
                    ] += 1

                    counted_down.add(
                        track_id
                    )

                    print(
                        f"DOWNWARD -> "
                        f"{vehicle_name} "
                        f"ID: {track_id}"
                    )


            # Save current position

            previous_positions[
                track_id
            ] = center_y


            # ==================================
            # DRAW BOUNDING BOX
            # ==================================

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # ==================================
            # DISPLAY VEHICLE NAME + ID
            # ==================================

            label = (
                f"{vehicle_name} "
                f"ID:{track_id}"
            )

            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )


    # ======================================
    # DISPLAY TOTAL COUNTS
    # ======================================

    cv2.putText(
        frame,
        f"UPWARD: {up_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )


    cv2.putText(
        frame,
        f"DOWNWARD: {down_count}",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )


    # ======================================
    # VEHICLE-WISE COUNTS
    # ======================================

    y = 120

    for vehicle in vehicle_classes.values():

        text = (
            f"{vehicle}: "
            f"UP {up_vehicle[vehicle]} | "
            f"DOWN {down_vehicle[vehicle]}"
        )

        cv2.putText(
            frame,
            text,
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

        y += 30


    # ======================================
    # SHOW VIDEO
    # ======================================

    cv2.imshow(
        "Vehicle Detection",
        frame
    )


    # Press Q to stop

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ==========================================
# RELEASE
# ==========================================

cap.release()

cv2.destroyAllWindows()


# ==========================================
# FINAL RESULT
# ==========================================

print("\n==============================")
print("       FINAL RESULT")
print("==============================")

print("\nUPWARD:")

for vehicle, count in up_vehicle.items():

    print(
        f"{vehicle}: {count}"
    )


print("\nDOWNWARD:")

for vehicle, count in down_vehicle.items():

    print(
        f"{vehicle}: {count}"
    )


print("\nTotal Upward:", up_count)

print("Total Downward:", down_count)